In [ ]:
# Add project root to Python path and change working directory
import sys
import os
from pathlib import Path

# Get project root
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))

# Change working directory to project root so config.yaml can be found
os.chdir(project_root)

print(f"Added to path: {project_root}")
print(f"Working directory: {os.getcwd()}")

### Docling API Test
---



In [ ]:
from src.preprocessing.document_parser import DocumentParser
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling_core.types.doc import PictureItem, TableItem

In [ ]:
# Configure Docling pipeline options
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA
)

pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = accelerator_options
pipeline_options.do_table_structure = True
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF, InputFormat.DOCX, InputFormat.HTML],
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)
file_path = "./src/data/pdf/AAPL/10-K/AAPL_10-K_0000320193-25-000079.pdf"
# file_path = "./src/data/pdf/sample/sample-unstructured-paper.pdf"
result = converter.convert(file_path)
doc = result.document

In [ ]:
from io import BytesIO
import base64
from PIL import Image

def pil_to_base64(image) -> str:
    """
        Convert a PIL Image to a base64 string.
        Args:
            image: PIL Image object
        Returns:
            Base64-encoded image string (utf-8)
    """

    buffer = BytesIO()
    image.save(buffer, format="PNG")

    img_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

    return img_base64


In [ ]:
from IPython.display import display, Markdown
from src.utils.models import load_vision_model

vision_client = load_vision_model()
tables_processed = 0
max_tables = 5

for idx, (item,level) in enumerate(doc.iterate_items()):
    # Extract page number from provenance
    page_number = None
    if hasattr(item, 'prov') and item.prov:
        page_refs = [p.page_no for p in item.prov if hasattr(p, 'page_no')]
        page_number = page_refs[0] if page_refs else None

    # Extract table image if item is a table
    if isinstance(item, TableItem):
        table_image = item.get_image(doc)
    
        display(table_image)
        
        table_md = item.export_to_markdown(doc)
        display(Markdown(table_md))

        prompt = (
            "Below is a markdown table extracted via OCR from a "
            "financial document. Compare it against the table image "
            "and correct any OCR errors, missing values, or "
            "formatting issues. Return ONLY the corrected markdown "
            f"table, nothing else.\n\n{table_md}"
        )
        response = vision_client.describe_image(table_image, prompt)
        display(Markdown(response))

        tables_processed += 1
        if tables_processed >= max_tables:
            break  # Stop after processing the desired number of tables
  

### Batch Processing
---


In [ ]:
from src.utils.logger import get_logger
from src.preprocessing.chunking import SemanticChunker
from src.indexing.vector_store import get_vector_store
from src.indexing.embedder import get_embedder
from src.utils.utils import extract_metadata_from_filename

logger = get_logger(__name__)

def get_pdf_files(folder_path: str) -> list[str]:
    """Get all PDF files from a folder as strings."""
    folder = Path(folder_path)
    return [str(pdf) for pdf in folder.glob("*.pdf")]


def parse_documents(
    file_paths: List[Path],
    company: Optional[str] = None,
    document_type: Optional[str] = None,
    filing_date: Optional[str] = None,
) -> List[Chunk]:

    logger.info(f"Parsing {len(file_paths)} document(s)...")
    
    # Determine if we're using automatic metadata extraction
    auto_extract = company is None or document_type is None or filing_date is None
    
    if auto_extract:
        logger.info("Using automatic metadata extraction from filenames")
        logger.info("Expected format: COMPANY_DOCTYPE_ACCESSION.pdf (e.g., AAPL_10-K_0000320193-25-000079.pdf)")
    
    # Prepare metadata for each file
    file_metadata = []
    valid_file_paths = []
    
    for file_path in file_paths:
        logger.info(f"Processing: {file_path.name}")
        
        # Extract or use provided metadata
        if auto_extract:
            try:
                metadata = extract_metadata_from_filename(str(file_path))
                doc_company = metadata["company"]
                doc_type = metadata["document_type"]
                doc_date = metadata["filing_date"]
                logger.info(f"  Extracted: {doc_company} | {doc_type} | {doc_date}")
            except ValueError as e:
                logger.error(f"  ✗ {e}")
                logger.warning(f"  Skipping {file_path.name}")
                continue
        else:
            doc_company = company
            doc_type = document_type
            doc_date = filing_date
        
        valid_file_paths.append(file_path)
        file_metadata.append({
            "company": doc_company,
            "document_type": doc_type,
            "filing_date": doc_date,
        })
    
    if not valid_file_paths:
        logger.error("No valid files to process")
        return []
    
    # Batch parse all documents
    parser = get_document_parser()
    logger.info(f"Batch parsing {len(valid_file_paths)} documents...")
    parsed_docs = parser.parse_documents_batch([str(fp) for fp in valid_file_paths])
    
    # Chunk each parsed document
    chunker = get_chunker()
    all_chunks = []
    
    for parsed_doc, metadata in zip(parsed_docs, file_metadata):
        chunks = chunker.chunk_document(
            parsed_doc=parsed_doc,
            company=metadata["company"],
            document_type=metadata["document_type"],
            filing_date=metadata["filing_date"],
        )
        all_chunks.extend(chunks)
        
        logger.info(
            f"  ✓ {parsed_doc.file_path.name}: {parsed_doc.metadata['num_elements']} elements → {len(chunks)} chunks"
        )
    
    logger.info(f"Total chunks created: {len(all_chunks)}")
    return all_chunks